In [2]:
"""
Rigorous verification: is GR additive (post-Newtonian) formula
the first-order Taylor expansion of WILL RG exact algebraic ratio?

Method:
1. Symbolic Taylor expansion of RG factor in (kappa_E^2, beta_E^2).
2. Direct algebraic comparison with GR formula.
3. Numerical comparison with GPS parameters.
"""

import sympy as sp

# ---------------------------------------------------------------
# PART 1: Symbolic setup
# ---------------------------------------------------------------
# Independent small parameters
kE2, bE2 = sp.symbols('kappa_E2 beta_E2', positive=True)

# Geometric ratio (large, order ~1)
rho = sp.symbols('rho', positive=True)  # rho = R_E / r_GPS  (about 0.24 for GPS)

# Derived (per closure kappa^2 = 2 beta^2 for circular orbit)
kGPS2 = kE2 * rho
bGPS2 = kE2 * rho / 2

# Tau-projections (relational proper-time invariants)
tau_E   = sp.sqrt(1 - kE2)   * sp.sqrt(1 - bE2)
tau_GPS = sp.sqrt(1 - kGPS2) * sp.sqrt(1 - bGPS2)

# Exact RG dimensionless shift coefficient (before multiplying by D*M)
RG_coeff_exact = 1 - tau_E / tau_GPS

# ---------------------------------------------------------------
# PART 2: Taylor expansion in small parameters
# ---------------------------------------------------------------
# Use a single bookkeeping epsilon: kE2 -> eps*kE2, bE2 -> eps*bE2
# (rho is NOT small; it is geometric)
eps = sp.symbols('eps', positive=True)
RG_eps = RG_coeff_exact.subs([(kE2, eps*kE2), (bE2, eps*bE2)])

# First-order term
RG_order1 = sp.series(RG_eps, eps, 0, 2).removeO().subs(eps, 1)
RG_order1 = sp.simplify(sp.expand(RG_order1))

# Second-order term (what GR misses)
RG_order2_full = sp.series(RG_eps, eps, 0, 3).removeO().subs(eps, 1)
RG_order2_only = sp.simplify(sp.expand(RG_order2_full - RG_order1))

print("="*70)
print("RG first-order expansion coefficient:")
print("="*70)
sp.pprint(RG_order1)

print("\n" + "="*70)
print("RG second-order (and higher) correction (what GR omits):")
print("="*70)
sp.pprint(RG_order2_only)

# ---------------------------------------------------------------
# PART 3: GR formula expressed in same variables
# ---------------------------------------------------------------
# Standard GR additive form:
#   Delta = (Phi_GPS - Phi_E)/c^2 - (v_GPS^2 - v_E^2)/(2 c^2)
# Using:
#   Phi/c^2 = -kappa^2 / 2   (since kappa^2 = 2GM/(rc^2))
#   v^2/c^2 = beta^2
GR_coeff = (-kGPS2/2 - (-kE2/2)) - (bGPS2 - bE2)/2
GR_coeff = sp.simplify(sp.expand(GR_coeff))

print("\n" + "="*70)
print("GR additive coefficient (same variables):")
print("="*70)
sp.pprint(GR_coeff)

# ---------------------------------------------------------------
# PART 4: Algebraic identity check
# ---------------------------------------------------------------
diff = sp.simplify(sp.expand(RG_order1 - GR_coeff))
print("\n" + "="*70)
print("RG_first_order  -  GR_coefficient  =")
print("="*70)
sp.pprint(diff)

if diff == 0:
    print("\n[VERIFIED] First-order Taylor expansion of WILL RG ≡ GR formula.")
else:
    print("\n[FAILED] Discrepancy remains; investigate.")

RG first-order expansion coefficient:
β_E2   3⋅κ_E2⋅ρ   κ_E2
──── - ──────── + ────
 2        4        2  

RG second-order (and higher) correction (what GR omits):
    2                                      2  2         2         2
β_E2    3⋅β_E2⋅κ_E2⋅ρ   β_E2⋅κ_E2   19⋅κ_E2 ⋅ρ    3⋅κ_E2 ⋅ρ   κ_E2 
───── + ───────────── - ───────── - ─────────── + ───────── + ─────
  8           8             4           32            8         8  

GR additive coefficient (same variables):
β_E2   3⋅κ_E2⋅ρ   κ_E2
──── - ──────── + ────
 2        4        2  

RG_first_order  -  GR_coefficient  =
0

[VERIFIED] First-order Taylor expansion of WILL RG ≡ GR formula.


In [3]:
"""
Numerical comparison: WILL RG (exact) vs GR (first-order)
for the Earth-GPS system. Quantifies the size of the higher-order
correction that GR omits.
"""

import mpmath as mp

# High precision
mp.mp.dps = 40

# Constants (CODATA/IERS-consistent values)
c        = mp.mpf('299792458')                # exact, m/s
GM_Earth = mp.mpf('3.986004418e14')           # m^3/s^2
R_Earth  = mp.mpf('6378137')                  # m (WGS84 equatorial)
r_GPS    = mp.mpf('26561750')                 # m (nominal GPS semi-major axis ~ 20200 km altitude)
D_ay     = mp.mpf('86400')                    # s
M_icro   = mp.mpf('1e6')

# Earth equatorial rotational velocity (sidereal would be 86164.0905;
# the problem statement uses D_ay = 86400, so we follow that convention).
v_Earth  = 2 * mp.pi * R_Earth / D_ay

# Circular GPS orbital velocity
v_GPS    = mp.sqrt(GM_Earth / r_GPS)

# Schwarzschild radius factor for Earth surface and GPS shell
kappa_E2   = 2 * GM_Earth / (R_Earth * c**2)
kappa_GPS2 = 2 * GM_Earth / (r_GPS    * c**2)

# Kinematic projections
beta_E2   = (v_Earth / c)**2
beta_GPS2 = (v_GPS   / c)**2

# Consistency check: for circular orbit, beta_GPS^2 should equal kappa_GPS^2 / 2
closure_residual = beta_GPS2 - kappa_GPS2 / 2

print("Input parameters (dimensionless):")
print(f"  kappa_E^2   = {mp.nstr(kappa_E2,   18)}")
print(f"  kappa_GPS^2 = {mp.nstr(kappa_GPS2, 18)}")
print(f"  beta_E^2    = {mp.nstr(beta_E2,    18)}")
print(f"  beta_GPS^2  = {mp.nstr(beta_GPS2,  18)}")
print(f"  closure residual (beta_GPS^2 - kappa_GPS^2/2) = {mp.nstr(closure_residual, 6)}")

# -----------------------------------------------------------
# WILL RG: exact algebraic ratio
# -----------------------------------------------------------
tau_E   = mp.sqrt(1 - kappa_E2)   * mp.sqrt(1 - beta_E2)
tau_GPS = mp.sqrt(1 - kappa_GPS2) * mp.sqrt(1 - beta_GPS2)

dt_RG = (1 - tau_E / tau_GPS) * D_ay * M_icro

# -----------------------------------------------------------
# GR: first-order post-Newtonian additive formula
# -----------------------------------------------------------
Phi_E   = -GM_Earth / R_Earth
Phi_GPS = -GM_Earth / r_GPS

dt_GR = ((Phi_GPS - Phi_E) / c**2
         - (v_GPS**2 - v_Earth**2) / (2 * c**2)
        ) * D_ay * M_icro

# -----------------------------------------------------------
# Theoretical second-order correction
# -----------------------------------------------------------
# From symbolic expansion:
#   delta_2 = beta_E^4/8 + 3*beta_E^2*kappa_E^2*rho/8 - beta_E^2*kappa_E^2/4
#             - 19*kappa_E^4*rho^2/32 + 3*kappa_E^4*rho/8 + kappa_E^4/8
rho = R_Earth / r_GPS
delta_2 = ( beta_E2**2 / 8
          + 3 * beta_E2 * kappa_E2 * rho / 8
          - beta_E2 * kappa_E2 / 4
          - mp.mpf(19) * kappa_E2**2 * rho**2 / 32
          + 3 * kappa_E2**2 * rho / 8
          + kappa_E2**2 / 8 )

dt_predicted_correction = delta_2 * D_ay * M_icro

# -----------------------------------------------------------
# Report
# -----------------------------------------------------------
print("\nResults (microseconds per day):")
print(f"  WILL RG exact :  {mp.nstr(dt_RG, 18)}")
print(f"  GR first-ord. :  {mp.nstr(dt_GR, 18)}")
print(f"  RG - GR       :  {mp.nstr(dt_RG - dt_GR, 6)}")
print(f"  predicted 2nd :  {mp.nstr(dt_predicted_correction, 6)}")
print(f"  agreement of (RG-GR) with predicted second-order term:")
print(f"      residual  :  {mp.nstr((dt_RG - dt_GR) - dt_predicted_correction, 6)}")

# Relative magnitude of the omitted term
rel = (dt_RG - dt_GR) / dt_RG
print(f"\nRelative size of GR omission: {mp.nstr(rel, 6)}  (~10^-10 of total shift)")

Input parameters (dimensionless):
  kappa_E^2   = 1.39069701360057669e-9
  kappa_GPS^2 = 3.33940951866324374e-10
  beta_E^2    = 2.39374857558587693e-12
  beta_GPS^2  = 1.66970475933162187e-10
  closure residual (beta_GPS^2 - kappa_GPS^2/2) = -5.34553e-51

Results (microseconds per day):
  WILL RG exact :  38.5421472752401783
  GR first-ord. :  38.5421472450724036
  RG - GR       :  3.01678e-8
  predicted 2nd :  3.01678e-8
  agreement of (RG-GR) with predicted second-order term:
      residual  :  2.21137e-17

Relative size of GR omission: 7.82722e-10  (~10^-10 of total shift)
